# Experiment 3 — Standalone Diagnostic: is `n_embd_tok` actually gating the override lookup?

Self-contained — no dependency on the original notebook's kernel state. Everything needed
(grammar, LangSpec, TinyGPT, tokenizer, training, eval, audit) is copied verbatim from
`Experiment_3_Number_Agreement_PCFG.ipynb` Sections 1–8, unchanged, followed by the diagnostic
itself (Section 12 from that notebook). Run top to bottom.

**Why this diagnostic, not a grammar redesign**: `TinyGPT` sweeps only `n_embd_tok` (the
initial token-embedding width). Every attention block, the 256-unit MLP, and the output head
run at a fixed `n_embd=64` regardless. The `(quantifier, noun_class) -> override` lookup is
resolved entirely inside that fixed-width machinery from two token identities that are each
individually cheap to keep separable even at tiny `n_embd_tok`. If that's right, no override
mechanism — whatever grammar it's embedded in — will show a capacity effect until the swept
variable actually gates the computation. This runs before any redesign, isolated-anchor-only
(no tag, no joint training), so the result is attributable to table size alone.

## 0. Setup

In [1]:
# GPU selection MUST happen before `import torch` -- CUDA's visible-device
# list is locked in at torch's CUDA-context init, so setting this after
# import silently does nothing. Change GPU_ID if your machine has more than
# one GPU and you want a specific one (leave as "0" for a single-GPU box).
import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import random, time, math, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(0)
    print('device: cuda ->', torch.cuda.get_device_name(0))
else:
    print('device: cpu (no GPU visible -- check GPU_ID / CUDA_VISIBLE_DEVICES if this is unexpected)')


device: cuda -> NVIDIA A100 80GB PCIe


## 1. Shared PCFG structure (verbatim from Section 1)

In [2]:
"""
Experiment 3 core: shared PCFG structure, per-language word-order/attachment
(tau) and quantifier-override rule (epsilon), surface realization, sampler.
"""
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

# ───────────────────────── Lexicon ─────────────────────────
# Two noun classes, shared across ALL languages (same stem -> same class
# everywhere). Class is what the override rule conditions on; it is NOT a
# per-language fact, so it cannot be a source of cross-language epsilon.
N_COUNT_NOUNS = 300
N_MASS_NOUNS  = 300
COUNT_NOUNS = [f"cnt{i:04d}" for i in range(N_COUNT_NOUNS)]
MASS_NOUNS  = [f"mss{i:04d}" for i in range(N_MASS_NOUNS)]
NOUN_CLASS = {n: 'count' for n in COUNT_NOUNS}
NOUN_CLASS.update({n: 'mass' for n in MASS_NOUNS})
ALL_NOUNS = COUNT_NOUNS + MASS_NOUNS

# Two shared quantifier tokens (closed class, same surface tokens in every
# language -- this is what makes the override-rule difference a genuine
# rule conflict rather than a different lookup table over disjoint tokens).
QUANTIFIERS = ['q_sev', 'q_man']   # stand-ins for e.g. "several" / "much"

N_VTR = 6
N_VINTR = 4
VERB_STEMS_TR = [f"vtr{i}" for i in range(N_VTR)]
VERB_STEMS_INTR = [f"vin{i}" for i in range(N_VINTR)]
PREPS = ['p0', 'p1', 'p2', 'p3']
RPRO = 'rp'
CONJ = 'cj'
DET = 'dt'

# ───────────────────────── PCFG production rules ─────────────────────────
# All probabilities are SHARED across every language -- word order and the
# override rule enter only in surface realization, never here. This mirrors
# the design principle from the earlier notebook: one structural grammar,
# many realizations.
P_S_COORD        = 0.20   # S -> CLAUSE CONJ CLAUSE
P_MOD_PRESENT    = 0.45   # NP -> DET N (MOD)
P_MOD_IS_REL     = 0.50   # MOD -> PP | RELCLAUSE
P_TRANSITIVE     = 0.55   # VP -> V_intr | V_tr NP_obj
P_OBJ_MOD        = 0.30   # object NP also optionally takes a MOD
P_QUANT_PRESENT  = 0.35   # NP -> (Q) DET N (MOD)  -- quantifier attaches to subject/obj NP
P_ATTRACTOR_PL   = 0.5    # attractor noun's own lexical number, sampled independently


def sample_number() -> str:
    return random.choice(['SG', 'PL'])


@dataclass
class NP:
    noun: str
    noun_class: str
    quant: Optional[str]
    lexical_number: str          # the noun's own base number, before override
    mod: Optional[dict]          # attractor info, or None
    final_number: str = None     # filled in by apply_override


def sample_np(rng: random.Random, allow_mod=True) -> NP:
    noun = rng.choice(ALL_NOUNS)
    ncls = NOUN_CLASS[noun]
    lex_num = sample_number()
    quant = rng.choice(QUANTIFIERS) if rng.random() < P_QUANT_PRESENT else None
    mod = None
    if allow_mod and rng.random() < P_MOD_PRESENT:
        attractor_noun = rng.choice(ALL_NOUNS)
        mod = dict(
            kind='PP' if rng.random() < P_MOD_IS_REL else 'REL',
            attractor_noun=attractor_noun,
            attractor_number=sample_number(),
        )
    return NP(noun=noun, noun_class=ncls, quant=quant, lexical_number=lex_num, mod=mod)


@dataclass
class Clause:
    subj: NP
    verb_stem: str
    transitive: bool
    obj: Optional[NP]


def sample_clause(rng: random.Random) -> Clause:
    subj = sample_np(rng, allow_mod=True)
    transitive = rng.random() < P_TRANSITIVE
    if transitive:
        obj = sample_np(rng, allow_mod=(rng.random() < P_OBJ_MOD))
        verb_stem = rng.choice(VERB_STEMS_TR)
    else:
        obj = None
        verb_stem = rng.choice(VERB_STEMS_INTR)
    return Clause(subj=subj, verb_stem=verb_stem, transitive=transitive, obj=obj)


@dataclass
class Sentence:
    clauses: List[Clause]
    coordinated: bool


def sample_sentence(rng: random.Random) -> Sentence:
    c1 = sample_clause(rng)
    if rng.random() < P_S_COORD:
        c2 = sample_clause(rng)
        return Sentence(clauses=[c1, c2], coordinated=True)
    return Sentence(clauses=[c1], coordinated=False)


## 2. Per-language parameters: τ (attachment) and ε (override rule) (verbatim from Section 2)

In [3]:
from dataclasses import dataclass
from typing import Dict, Tuple
import random


OVERRIDE_CELLS = [(q, c) for q in QUANTIFIERS for c in ('count', 'mass')]
# Anchor rule: q_sev forces PL on count nouns, no effect on mass nouns;
# q_man forces SG on mass nouns, no effect on count nouns. This is a
# genuine conditional rule (depends on class, not just quantifier identity).
ANCHOR_OVERRIDE: Dict[Tuple[str, str], str] = {
    ('q_sev', 'count'): 'PL',
    ('q_sev', 'mass'):  'NONE',
    ('q_man', 'count'): 'NONE',
    ('q_man', 'mass'):  'SG',
}


def build_override_table(eps_flips: int, seed: int) -> Dict[Tuple[str, str], str]:
    rng = random.Random(seed)
    table = dict(ANCHOR_OVERRIDE)
    cells = list(OVERRIDE_CELLS)
    rng.shuffle(cells)
    outcomes = ['NONE', 'SG', 'PL']
    for cell in cells[:eps_flips]:
        current = table[cell]
        table[cell] = rng.choice([o for o in outcomes if o != current])
    return table


@dataclass
class LangSpec:
    name: str
    tau_intervene: float   # P(in-situ MOD attachment | MOD present)
    eps_flips: int          # 0..4
    seed: int
    anchor: bool = False

    def __post_init__(self):
        self.override_table = (dict(ANCHOR_OVERRIDE) if self.anchor
                                else build_override_table(self.eps_flips, self.seed))

    def apply_override(self, np_: NP) -> str:
        """Compute the FINAL number the verb must agree with for this NP,
        in THIS language: lexical number unless a quantifier + this
        language's rule table forces an override."""
        if np_.quant is None:
            return np_.lexical_number
        forced = self.override_table[(np_.quant, np_.noun_class)]
        return np_.lexical_number if forced == 'NONE' else forced

    def mod_in_situ(self, rng: random.Random) -> bool:
        return rng.random() < self.tau_intervene


## 3. Surface realization (verbatim from Section 3)

In [4]:
import random
from typing import List, Dict, Tuple


BOS, EOS = '<BOS>', '<EOS>'
DOT = '.'
DET = 'dt'  # unused directly; determiner now carries lexical number (see realize_np)
DET_SG, DET_PL = 'dt_sg', 'dt_pl'

# Shared, language-tagged agreement-bearing verb forms (SG/PL), so a
# cross-language accuracy difference can't be explained by relearning
# separate output lexica (same design principle as the earlier notebook).
VFORMS = {(stem, n): f"agr_{stem}_{n}" for stem in VERB_STEMS_TR + VERB_STEMS_INTR
          for n in ('SG', 'PL')}


def realize_np(np_: NP, lang: LangSpec, role: str) -> Tuple[List[str], str]:
    """Returns (tokens, final_number). Quantifier/DET/Noun in order; MOD
    (if any) is placed IN-SITU (right after the noun) or held back for
    extraposition by the caller, per lang.tau_intervene.

    The determiner surface-marks the noun's own LEXICAL number (dt_sg /
    dt_pl) -- this is the only place that information is readable from the
    tokens at all, so the "no quantifier" case is genuinely learnable (copy
    the marked default). A quantifier + this language's override rule can
    then force the AGREEMENT to differ from that marked default -- e.g.
    "several <dt_sg> cattle" still surface-marked singular, but agreement
    is forced plural by the quantifier rule in some languages. That
    mismatch-when-overridden is exactly the genuine, non-memorizable
    computation we want (the determiner alone is no longer sufficient)."""
    toks = []
    if np_.quant:
        toks.append(np_.quant)
    det = DET_PL if np_.lexical_number == 'PL' else DET_SG
    toks += [det, np_.noun]
    final_number = lang.apply_override(np_)
    return toks, final_number


def realize_mod(np_: NP, rng: random.Random) -> List[str]:
    mod = np_.mod
    if mod is None:
        return []
    det = DET_PL if mod['attractor_number'] == 'PL' else DET_SG
    if mod['kind'] == 'PP':
        prep = rng.choice(['p0', 'p1', 'p2', 'p3'])
        return [prep, det, mod['attractor_noun']]
    else:
        return ['rp', det, mod['attractor_noun']]  # simplified relative clause


def realize_clause(clause: Clause, lang: LangSpec, rng: random.Random, ann: List[dict], toks: List[str]):
    subj_toks, subj_final_num = realize_np(clause.subj, lang, role='subj')
    subj_mod_toks = realize_mod(clause.subj, rng)
    mod_in_situ = lang.mod_in_situ(rng) if clause.subj.mod is not None else False

    toks += subj_toks
    if clause.subj.mod is not None and mod_in_situ:
        toks += subj_mod_toks   # attractor sits BETWEEN subject and verb

    # ---- verb slot: this is the prediction target ----
    verb_pos = len(toks)  # position of the (about-to-be-appended) verb token
    vform = VFORMS[(clause.verb_stem, subj_final_num)]
    toks.append(vform)
    ann.append(dict(pos=verb_pos, subcomp='subj_verb_agree',
                     target_number=subj_final_num,
                     had_intervening_attractor=bool(clause.subj.mod is not None and mod_in_situ),
                     attractor_mismatches=(clause.subj.mod is not None and mod_in_situ and
                                           clause.subj.mod['attractor_number'] != subj_final_num)))

    if clause.subj.mod is not None and not mod_in_situ:
        toks += subj_mod_toks   # extraposed: AFTER the verb, doesn't intervene

    if clause.transitive and clause.obj is not None:
        obj_toks, _ = realize_np(clause.obj, lang, role='obj')
        obj_mod_toks = realize_mod(clause.obj, rng)
        toks += obj_toks
        if clause.obj.mod is not None:
            toks += obj_mod_toks  # object modifiers never intervene in subj-verb agreement; harmless filler


def realize_sentence(sent: Sentence, lang: LangSpec, rng: random.Random) -> Tuple[List[str], List[dict]]:
    toks = [BOS]
    ann = []
    for i, clause in enumerate(sent.clauses):
        realize_clause(clause, lang, rng, ann, toks)
        if i == 0 and sent.coordinated:
            toks.append('cj')
    toks.append(DOT)
    toks.append(EOS)
    return toks, ann


## 4. Tokenizer / shared vocabulary (verbatim from Section 4)

In [5]:
"""Tokenizer + batched example generation for a given language."""
import random
import torch
from typing import List

# ---- vocabulary (shared across ALL languages: same token = same id everywhere) ----
SPECIAL = ['<PAD>', BOS, EOS]
VOCAB = SPECIAL + [DOT, DET_SG, DET_PL, 'rp', 'cj'] + PREPS + QUANTIFIERS + ALL_NOUNS + list(VFORMS.values())
VOCAB = list(dict.fromkeys(VOCAB))  # dedupe, preserve order
STOI = {t: i for i, t in enumerate(VOCAB)}
ITOS = {i: t for t, i in STOI.items()}
PAD_ID = STOI['<PAD>']
VOCAB_SIZE = len(VOCAB)

# restricted-vocab feature eval: at an agreement position, the model only
# needs to choose between the SG/PL forms of the verb stem actually used.
VFORM_SG_IDS = {stem: STOI[VFORMS[(stem, 'SG')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}
VFORM_PL_IDS = {stem: STOI[VFORMS[(stem, 'PL')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}


def make_example(lang: LangSpec, rng: random.Random, max_len: int):
    s = sample_sentence(rng)
    toks, ann = realize_sentence(s, lang, rng)
    if len(toks) > max_len:
        return None
    ids = [STOI[t] for t in toks]
    return ids, ann, toks


def make_batch(lang: LangSpec, rng: random.Random, batch_size: int, max_len: int = 24):
    seqs, anns = [], []
    while len(seqs) < batch_size:
        ex = make_example(lang, rng, max_len)
        if ex is None:
            continue
        ids, ann, toks = ex
        seqs.append(ids)
        anns.append(ann)
    x = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return x, anns


## 5. Model: capacity-bottlenecked TinyGPT (verbatim from Section 5)

In [6]:
"""Tiny decoder-only transformer with a separate token-identity bottleneck
(n_embd_tok) from the residual width (n_embd)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd_tok, n_embd=64, n_layer=3, n_head=4, max_len=24, pad_id=0):
        super().__init__()
        self.tok_bottleneck = nn.Embedding(vocab_size, n_embd_tok, padding_idx=pad_id)
        self.tok_proj = nn.Linear(n_embd_tok, n_embd) if n_embd_tok != n_embd else nn.Identity()
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_proj(self.tok_bottleneck(idx)) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)


## 6. Training + feature-level evaluation (verbatim from Section 6)

In [7]:
import random
import torch
import torch.nn.functional as F
from typing import List


def compute_loss(model, x, device):
    x = x.to(device)
    logits = model(x[:, :-1])
    targets = x[:, 1:].contiguous()
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                            ignore_index=PAD_ID)
    return loss


def train_isolated(lang: LangSpec, n_embd_tok, steps=1500, seed=0, lr=2e-2,
                    n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu'):
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    for step in range(steps):
        opt.zero_grad()
        x, _ = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss(model, x, device)
        loss.backward()
        opt.step()
        sched.step()
    return model


@torch.no_grad()
def feature_accuracy(model, lang: LangSpec, device='cpu', n_eval=4000, max_len=24,
                      seed=999, split_by_intervention=False):
    """Restricted-vocab accuracy at subj-verb agreement positions: does the
    model prefer the correct-number verb FORM over the wrong-number form of
    the SAME stem (so full-vocab lemma-choice noise never enters this
    number)."""
    model.eval()
    rng = random.Random(seed)
    correct = {'all': 0}
    total = {'all': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        logits = model(x[:, :-1])
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']
                if pos == 0 or pos >= x.shape[1]:
                    continue
                stem_token_id = x[i, pos].item()
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                correct['all'] += is_correct
                total['all'] += 1
                if split_by_intervention:
                    key = 'intervene' if a['had_intervening_attractor'] else 'no_intervene'
                    correct.setdefault(key, 0); total.setdefault(key, 0)
                    correct[key] += is_correct; total[key] += 1
        n_done += bs
    return {k: correct[k] / total[k] for k in total}


## 7. Leakage / calibration audit (verbatim from Section 8)

In [8]:
"""Leakage / calibration audit: verify twin pairs differ in EXACTLY the
intended parameter, and report the base rates that make tau/eps
manipulations real (not degenerate)."""
import random


def audit_twin_invariance(lang_x: LangSpec, lang_y: LangSpec, expected_diff: str):
    diffs = []
    if lang_x.tau_intervene != lang_y.tau_intervene:
        diffs.append('tau_intervene')
    if lang_x.override_table != lang_y.override_table:
        diffs.append('override_table')
    ok = (diffs == ['tau_intervene'] if expected_diff == 'tau' else diffs == ['override_table'])
    return dict(ok=ok, diffs=diffs)


def audit_rates(lang: LangSpec, n=20000, seed=0):
    rng = random.Random(seed)
    n_agree = n_intervene = n_mismatch = n_quant = n_override_active = 0
    for _ in range(n):
        s = sample_sentence(rng)
        toks, ann = realize_sentence(s, lang, rng)
        for a in ann:
            n_agree += 1
            n_intervene += int(a['had_intervening_attractor'])
            n_mismatch += int(a['attractor_mismatches'])
        for cl in s.clauses:
            if cl.subj.quant is not None:
                n_quant += 1
                if lang.apply_override(cl.subj) != cl.subj.lexical_number:
                    n_override_active += 1
    return dict(
        intervene_rate=n_intervene / n_agree,
        recency_would_be_wrong_rate=n_mismatch / n_agree,
        quant_present_rate=n_quant / n_agree,
        override_changes_answer_rate=(n_override_active / n_quant) if n_quant else 0.0,
    )


## 8. Diagnostic: does the override lookup actually compete for `n_embd_tok`?

`n_embd_tok` is the *only* swept capacity variable above — every attention block, the
256-unit MLP, and the output head run at a **fixed** `n_embd=64` regardless. The
`(quantifier, noun_class) -> override` lookup is resolved entirely inside that fixed-width
machinery, from two token identities each individually cheap to keep separable even at tiny
`n_embd_tok`. If that's right, no table size will ever show a capacity effect. This settles
it directly: isolated-anchor-only training (no tag, no joint training) at three table sizes,
across the same capacity grid used throughout the project.

In [9]:
"""
Rebuilds QUANTIFIERS / noun classes / the override table at a chosen size, then rebuilds the
tokenizer globals so the new tokens exist in vocab. Everything else (sampler, surface
realization, TinyGPT, train_isolated, feature_accuracy, LangSpec, audit_rates) is untouched
and reused as-is -- this ONLY changes table size, so any effect we see is attributable to
table size alone.

Class remains a fixed lexical fact about the same 600 noun stems, shared across every
language -- only the RULE (anchor override table) grows; which class a noun belongs to is
still assigned once, structurally, not learned or varied by language.
"""
import random as _random

def build_big_table_globals(n_quant: int, n_class: int, anchor_seed: int = 12345):
    """Overwrites QUANTIFIERS / NOUN_CLASS / OVERRIDE_CELLS / ANCHOR_OVERRIDE / VOCAB /
    STOI / ITOS / VOCAB_SIZE / VFORM_SG_IDS / VFORM_PL_IDS / PAD_ID in place (module
    globals). The existing pipeline reads these as globals, not passed-in args, so it
    picks up the new values automatically -- no other function needs touching.
    """
    global QUANTIFIERS, NOUN_CLASS, OVERRIDE_CELLS, ANCHOR_OVERRIDE
    global VOCAB, STOI, ITOS, VOCAB_SIZE, VFORM_SG_IDS, VFORM_PL_IDS, PAD_ID

    QUANTIFIERS = [f"q{i}" for i in range(n_quant)]

    classes = [f"cls{i}" for i in range(n_class)]
    NOUN_CLASS = {}
    for idx, noun in enumerate(ALL_NOUNS):
        NOUN_CLASS[noun] = classes[idx % n_class]

    OVERRIDE_CELLS = [(q, c) for q in QUANTIFIERS for c in classes]

    # Deterministic (seeded) anchor table. ~50% NONE / 25%/25% SG/PL, matching the
    # original 4-cell table's rough ratio (2 of 4 were NONE), so
    # override_changes_answer_rate stays non-degenerate at bigger sizes too -- CONFIRM
    # with audit_rates() below rather than assuming it holds.
    rng = _random.Random(anchor_seed)
    outcomes_weighted = ['NONE', 'NONE', 'SG', 'PL']
    ANCHOR_OVERRIDE = {cell: rng.choice(outcomes_weighted) for cell in OVERRIDE_CELLS}

    # Rebuild tokenizer globals -- new quantifier tokens must exist in vocab or
    # make_batch will KeyError the first time one is sampled.
    VOCAB = SPECIAL + [DOT, DET_SG, DET_PL, 'rp', 'cj'] + PREPS + QUANTIFIERS + ALL_NOUNS + list(VFORMS.values())
    VOCAB = list(dict.fromkeys(VOCAB))
    STOI = {t: i for i, t in enumerate(VOCAB)}
    ITOS = {i: t for t, i in STOI.items()}
    PAD_ID = STOI['<PAD>']
    VOCAB_SIZE = len(VOCAB)
    VFORM_SG_IDS = {stem: STOI[VFORMS[(stem, 'SG')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}
    VFORM_PL_IDS = {stem: STOI[VFORMS[(stem, 'PL')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}

    print(f"rebuilt globals: n_quant={n_quant} n_class={n_class} "
          f"table_size={len(OVERRIDE_CELLS)} vocab_size={VOCAB_SIZE}")


### 8a. Sanity check: rebuild machinery reproduces the known 4-cell result

If this doesn't come back near-ceiling, the rebuild has a bug — don't trust anything below
until this passes.

In [10]:
build_big_table_globals(n_quant=2, n_class=2, anchor_seed=12345)
A_sanity = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
print('audit:', audit_rates(A_sanity, n=8000))
for cap in [4, 8, 16, 32]:
    m = train_isolated(A_sanity, n_embd_tok=cap, steps=2000, seed=0, device=DEVICE)
    acc = feature_accuracy(m, A_sanity, device=DEVICE, n_eval=3000)['all']
    print(f"cap={cap:4d}  acc={acc:.4f}")
# Expect: near-ceiling (>0.95) even at cap=4, matching the known 4-cell behavior.


rebuilt globals: n_quant=2 n_class=2 table_size=4 vocab_size=634
audit: {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.013447305326800793, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.3848214285714286}
cap=   4  acc=0.9739
cap=   8  acc=0.9883
cap=  16  acc=0.9958
cap=  32  acc=0.9664


### 8b. The actual diagnostic: isolated accuracy vs. capacity, at three table sizes

In [11]:
import time

DIAG_CAPACITY_GRID = [4, 8, 16, 32, 64, 96]
DIAG_SEEDS = (0, 1, 2)
DIAG_STEPS = 3000  # match the main sweep's step count
TABLE_SIZES = [(2, 2), (8, 8), (16, 16)]  # table sizes 4, 64, 256

diag_rows = []
t0 = time.time()
for n_quant, n_class in TABLE_SIZES:
    build_big_table_globals(n_quant, n_class, anchor_seed=12345)
    A_diag = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
    rates = audit_rates(A_diag, n=8000)
    print(f"  audit ({n_quant}x{n_class}): {rates}")
    for cap in DIAG_CAPACITY_GRID:
        accs = []
        for seed in DIAG_SEEDS:
            m = train_isolated(A_diag, n_embd_tok=cap, steps=DIAG_STEPS, seed=seed, device=DEVICE)
            acc = feature_accuracy(m, A_diag, device=DEVICE, n_eval=3000)['all']
            accs.append(acc)
        mean_acc = sum(accs) / len(accs)
        diag_rows.append(dict(table=f"{n_quant}x{n_class}", n_cells=n_quant*n_class,
                               capacity=cap, acc=mean_acc, seeds=accs))
        print(f"[{time.time()-t0:6.0f}s] table={n_quant}x{n_class:<3} cap={cap:4d}  "
              f"acc={mean_acc:.4f}  (seeds={[f'{a:.3f}' for a in accs]})")

with open('exp3_diagnostic_rows.pkl', 'wb') as f:
    pickle.dump(diag_rows, f)
print("saved exp3_diagnostic_rows.pkl -- safe to reload if the kernel dies again:")
print("  diag_rows = pickle.load(open('exp3_diagnostic_rows.pkl', 'rb'))")


rebuilt globals: n_quant=2 n_class=2 table_size=4 vocab_size=634
  audit (2x2): {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.01240487855728135, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.3898809523809524}
[   169s] table=2x2   cap=   4  acc=0.9435  (seeds=['0.994', '0.904', '0.933'])
[   341s] table=2x2   cap=   8  acc=0.9903  (seeds=['0.993', '0.999', '0.978'])
[   507s] table=2x2   cap=  16  acc=0.9887  (seeds=['1.000', '0.967', '0.999'])
[   685s] table=2x2   cap=  32  acc=0.9711  (seeds=['0.992', '0.999', '0.923'])
[   859s] table=2x2   cap=  64  acc=0.9607  (seeds=['0.963', '0.923', '0.996'])
[  1030s] table=2x2   cap=  96  acc=0.9954  (seeds=['0.996', '0.999', '0.991'])
rebuilt globals: n_quant=8 n_class=8 table_size=64 vocab_size=640
  audit (8x8): {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.012717606588137184, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.20

### 8c. How to read the result

- **Flat, near-ceiling (>0.95) at every capacity, for every table size including 16x16=256**:
  the override lookup is being absorbed by the untouched wide MLP/head regardless of table
  size. Enlarging the table is not the fix — sweep `n_embd` (and MLP width) instead, and
  re-run this same diagnostic against *that* sweep before trusting it either.
- **Flat/near-ceiling at 4 cells (replicating the known result) but a real, capacity-dependent
  climb at 64 and/or 256 cells** (low accuracy at cap=4–8, rising toward ceiling by
  cap=64–96): the lookup *does* compete for `n_embd_tok` once the table is big enough.
  Enlarging the table is the right lever — and this sweep tells you roughly what size to use
  for the real eps sweep.
- **Never reaches ceiling even at cap=96 for the biggest table**: that table is too big to
  learn in this training budget — raise `DIAG_STEPS` before concluding anything; this is a
  training-budget confound, not a capacity-crossover result.

Whatever the outcome, this also needs re-running once the missing `train_joint` /
`make_batch_single_tagged` code is found or rewritten — a joint bilingual model could behave
differently from an isolated one even if isolated accuracy alone looks capacity-dependent
here, since joint training adds the tag/language-identity dimension this diagnostic
deliberately excludes.

In [12]:
import random, time
import torch

def train_isolated_long(lang, n_embd_tok, steps, seed=0, lr=2e-2, n_embd=64, n_layer=3,
                         n_head=4, max_len=24, device='cpu', eval_every=2000, n_eval=3000):
    """Same training loop as train_isolated, but logs loss + held-out accuracy every
    eval_every steps -- the thing we actually need to tell 'still improving, needs more
    steps' apart from 'genuinely plateaued', which a single final-accuracy number
    (as in Section 8) can't distinguish."""
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    running_loss = 0.0
    log = []
    t0 = time.time()
    for step in range(1, steps + 1):
        opt.zero_grad()
        x, _ = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss(model, x, device)
        loss.backward()
        opt.step()
        sched.step()
        running_loss += loss.item()
        if step % eval_every == 0:
            avg_loss = running_loss / eval_every
            acc = feature_accuracy(model, lang, device=device, n_eval=n_eval)['all']
            log.append(dict(step=step, loss=avg_loss, acc=acc, t=time.time() - t0))
            print(f"[{time.time()-t0:6.0f}s] step={step:6d}  loss={avg_loss:.4f}  acc={acc:.4f}")
            running_loss = 0.0
            model.train()
    return model, log


build_big_table_globals(n_quant=16, n_class=16, anchor_seed=12345)
A_long = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
rates = audit_rates(A_long, n=8000)
give_up_floor = 1 - rates['quant_present_rate'] * rates['override_changes_answer_rate']
print(f"audit: {rates}")
print(f"give-up floor (ignore quantifier entirely): {give_up_floor:.4f}")
print(f"(Section 8 saw ~0.90-0.92 at every capacity for this table -- right on this floor)\n")

LONG_STEPS = 20000
LONG_SEEDS = (0, 1)
long_logs = {}
for seed in LONG_SEEDS:
    print(f"--- seed {seed} ---")
    model, log = train_isolated_long(A_long, n_embd_tok=96, steps=LONG_STEPS, seed=seed,
                                      device=DEVICE, eval_every=2000, n_eval=3000)
    long_logs[seed] = log

import pickle
with open('exp3_long_training_check.pkl', 'wb') as f:
    pickle.dump(long_logs, f)
print("\nsaved exp3_long_training_check.pkl")

rebuilt globals: n_quant=16 n_class=16 table_size=256 vocab_size=648
audit: {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.010841238403002188, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.2193452380952381}
give-up floor (ignore quantifier entirely): 0.9232
(Section 8 saw ~0.90-0.92 at every capacity for this table -- right on this floor)

--- seed 0 ---
[    43s] step=  2000  loss=2.8165  acc=0.9173
[    83s] step=  4000  loss=2.7890  acc=0.8740
[   124s] step=  6000  loss=2.8007  acc=0.8685
[   164s] step=  8000  loss=2.7625  acc=0.8818
[   203s] step= 10000  loss=2.7489  acc=0.8876
[   246s] step= 12000  loss=2.7417  acc=0.8996
[   288s] step= 14000  loss=2.7303  acc=0.9046
[   330s] step= 16000  loss=2.7222  acc=0.8987
[   370s] step= 18000  loss=2.7169  acc=0.9120
[   410s] step= 20000  loss=2.7134  acc=0.9059
--- seed 1 ---
[    40s] step=  2000  loss=2.7955  acc=0.9165
[    80s] step=  4000  loss=2.7743  acc=0.9120
[   121

In [13]:
import random, time
import torch
import torch.nn.functional as F

def realize_clause(clause: Clause, lang: LangSpec, rng: random.Random, ann: List[dict], toks: List[str]):
    """Redefines Section 3's realize_clause to add one field: override_fired --
    whether the quantifier override actually changed the answer away from the
    noun's own marked default at THIS agreement position. Needed so the loss
    can be reweighted toward exactly the cases the model is currently
    ignoring."""
    subj_toks, subj_final_num = realize_np(clause.subj, lang, role='subj')
    subj_mod_toks = realize_mod(clause.subj, rng)
    mod_in_situ = lang.mod_in_situ(rng) if clause.subj.mod is not None else False

    toks += subj_toks
    if clause.subj.mod is not None and mod_in_situ:
        toks += subj_mod_toks

    verb_pos = len(toks)
    vform = VFORMS[(clause.verb_stem, subj_final_num)]
    toks.append(vform)
    override_fired = (clause.subj.quant is not None and subj_final_num != clause.subj.lexical_number)
    ann.append(dict(pos=verb_pos, subcomp='subj_verb_agree',
                     target_number=subj_final_num,
                     had_intervening_attractor=bool(clause.subj.mod is not None and mod_in_situ),
                     attractor_mismatches=(clause.subj.mod is not None and mod_in_situ and
                                           clause.subj.mod['attractor_number'] != subj_final_num),
                     override_fired=override_fired))

    if clause.subj.mod is not None and not mod_in_situ:
        toks += subj_mod_toks

    if clause.transitive and clause.obj is not None:
        obj_toks, _ = realize_np(clause.obj, lang, role='obj')
        obj_mod_toks = realize_mod(clause.obj, rng)
        toks += obj_toks
        if clause.obj.mod is not None:
            toks += obj_mod_toks


def compute_loss_weighted(model, x, anns, device, override_weight):
    """Same next-token cross-entropy as compute_loss, but agreement positions
    where override_fired get `override_weight`x the gradient weight of every
    other position (including ordinary agreement positions where the
    override didn't fire). Everything else -- nouns, determiners, quantifier
    tokens, non-firing agreement positions -- keeps weight 1.0, so this only
    touches the exact thing the model has been ignoring."""
    x = x.to(device)
    logits = model(x[:, :-1])
    targets = x[:, 1:].contiguous()
    B, Tm1 = targets.shape
    per_tok = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                               ignore_index=PAD_ID, reduction='none').view(B, Tm1)
    weight = torch.ones(B, Tm1, device=device)
    for i, ann_list in enumerate(anns):
        for a in ann_list:
            pos = a['pos']
            if pos == 0 or pos >= x.shape[1]:
                continue
            if a.get('override_fired', False):
                weight[i, pos - 1] = override_weight
    pad_mask = (targets != PAD_ID).float()
    weight = weight * pad_mask
    return (per_tok * weight).sum() / weight.sum().clamp_min(1.0)


def train_isolated_weighted(lang, n_embd_tok, steps, override_weight, seed=0, lr=2e-2,
                             n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu',
                             eval_every=2000, n_eval=3000):
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    running_loss = 0.0
    log = []
    t0 = time.time()
    for step in range(1, steps + 1):
        opt.zero_grad()
        x, anns = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss_weighted(model, x, anns, device, override_weight)
        loss.backward()
        opt.step()
        sched.step()
        running_loss += loss.item()
        if step % eval_every == 0:
            avg_loss = running_loss / eval_every
            acc = feature_accuracy(model, lang, device=device, n_eval=n_eval)['all']
            log.append(dict(step=step, loss=avg_loss, acc=acc, t=time.time() - t0))
            print(f"[{time.time()-t0:6.0f}s] step={step:6d}  loss={avg_loss:.4f}  acc={acc:.4f}")
            running_loss = 0.0
            model.train()
    return model, log


build_big_table_globals(n_quant=16, n_class=16, anchor_seed=12345)
A_rw = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
rates = audit_rates(A_rw, n=8000)
frac_override = rates['quant_present_rate'] * rates['override_changes_answer_rate']
give_up_floor = 1 - frac_override
# weight that roughly equalizes total gradient mass between override-firing and
# non-firing agreement positions -- a principled starting point, not a tuned knob
override_weight = round((1 - frac_override) / frac_override, 1)
print(f"audit: {rates}")
print(f"fraction of agreement decisions where override fires: {frac_override:.4f}")
print(f"give-up floor: {give_up_floor:.4f}  |  override_weight (mass-equalizing): {override_weight}")

RW_STEPS = 20000
RW_SEEDS = (0, 1)
rw_logs = {}
for seed in RW_SEEDS:
    print(f"--- seed {seed} ---")
    model, log = train_isolated_weighted(A_rw, n_embd_tok=96, steps=RW_STEPS,
                                          override_weight=override_weight, seed=seed,
                                          device=DEVICE, eval_every=2000, n_eval=3000)
    rw_logs[seed] = log

import pickle
with open('exp3_reweighted_check.pkl', 'wb') as f:
    pickle.dump(rw_logs, f)
print("\nsaved exp3_reweighted_check.pkl")

rebuilt globals: n_quant=16 n_class=16 table_size=256 vocab_size=648
audit: {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.013030334618993016, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.2226190476190476}
fraction of agreement decisions where override fires: 0.0780
give-up floor: 0.9220  |  override_weight (mass-equalizing): 11.8
--- seed 0 ---
[    42s] step=  2000  loss=2.8809  acc=0.6915
[    83s] step=  4000  loss=2.8367  acc=0.7375
[   124s] step=  6000  loss=2.8116  acc=0.7037
[   165s] step=  8000  loss=2.7985  acc=0.6998
[   207s] step= 10000  loss=2.7639  acc=0.7134
[   247s] step= 12000  loss=2.7558  acc=0.7567
[   289s] step= 14000  loss=2.7435  acc=0.7680
[   330s] step= 16000  loss=2.7330  acc=0.7916
[   367s] step= 18000  loss=2.7240  acc=0.7958
[   409s] step= 20000  loss=2.7177  acc=0.7983
--- seed 1 ---
[    40s] step=  2000  loss=2.8962  acc=0.5028
[    81s] step=  4000  loss=2.8284  acc=0.5058
[   123s] step=

In [14]:
import torch

@torch.no_grad()
def feature_accuracy_split(model, lang, device='cpu', n_eval=4000, max_len=24, seed=999):
    """Same restricted-vocab agreement check as feature_accuracy, but split into
    'override_fired' (the rare, hard cases -- this is the number that actually
    tells us whether the rule is being learned) vs 'no_override' (the easy 92%
    the model already had for free). Aggregate accuracy alone can't distinguish
    'learning the rule at some cost to the easy cases' from 'getting worse'."""
    model.eval()
    rng = random.Random(seed)
    correct = {'fired': 0, 'not_fired': 0}
    total = {'fired': 0, 'not_fired': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        logits = model(x[:, :-1])
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']
                if pos == 0 or pos >= x.shape[1]:
                    continue
                stem_token_id = x[i, pos].item()
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                key = 'fired' if a.get('override_fired', False) else 'not_fired'
                correct[key] += is_correct
                total[key] += 1
        n_done += bs
    out = {}
    for k in total:
        out[k] = correct[k] / total[k] if total[k] else float('nan')
    out['n_fired'] = total['fired']
    out['n_not_fired'] = total['not_fired']
    return out


def train_isolated_weighted_split(lang, n_embd_tok, steps, override_weight, seed=0, lr=2e-2,
                                   n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu',
                                   eval_every=2000, n_eval=4000):
    """Same as train_isolated_weighted, but logs the split metric at each
    checkpoint instead of aggregate accuracy -- override-case accuracy climbing
    toward the 50% chance floor and beyond is the thing that tells us the fix
    is working; non-override accuracy staying high tells us it isn't happening
    at the expense of the easy cases."""
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    running_loss = 0.0
    log = []
    t0 = time.time()
    for step in range(1, steps + 1):
        opt.zero_grad()
        x, anns = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss_weighted(model, x, anns, device, override_weight)
        loss.backward()
        opt.step()
        sched.step()
        running_loss += loss.item()
        if step % eval_every == 0:
            avg_loss = running_loss / eval_every
            split = feature_accuracy_split(model, lang, device=device, n_eval=n_eval)
            log.append(dict(step=step, loss=avg_loss, **split, t=time.time() - t0))
            print(f"[{time.time()-t0:6.0f}s] step={step:6d}  loss={avg_loss:.4f}  "
                  f"fired_acc={split['fired']:.4f} (n={split['n_fired']})  "
                  f"not_fired_acc={split['not_fired']:.4f} (n={split['n_not_fired']})")
            running_loss = 0.0
            model.train()
    return model, log


RW2_STEPS = 40000
RW2_SEEDS = (0, 1)
rw2_logs = {}
for seed in RW2_SEEDS:
    print(f"--- seed {seed} ---")
    model, log = train_isolated_weighted_split(A_rw, n_embd_tok=96, steps=RW2_STEPS,
                                                override_weight=override_weight, seed=seed,
                                                device=DEVICE, eval_every=2000, n_eval=4000)
    rw2_logs[seed] = log

import pickle
with open('exp3_reweighted_split_check.pkl', 'wb') as f:
    pickle.dump(rw2_logs, f)
print("\nsaved exp3_reweighted_split_check.pkl")

--- seed 0 ---
[    31s] step=  2000  loss=2.9113  fired_acc=0.5039 (n=383)  not_fired_acc=0.4988 (n=4415)
[    62s] step=  4000  loss=2.9321  fired_acc=0.4703 (n=370)  not_fired_acc=0.5070 (n=4428)
[    96s] step=  6000  loss=2.9111  fired_acc=0.7909 (n=397)  not_fired_acc=0.6973 (n=4401)
[   130s] step=  8000  loss=2.9295  fired_acc=0.8648 (n=392)  not_fired_acc=0.6700 (n=4406)
[   164s] step= 10000  loss=2.9198  fired_acc=0.6763 (n=380)  not_fired_acc=0.6510 (n=4418)
[   197s] step= 12000  loss=2.8805  fired_acc=0.8995 (n=398)  not_fired_acc=0.6714 (n=4400)
[   230s] step= 14000  loss=2.8929  fired_acc=0.6870 (n=409)  not_fired_acc=0.4723 (n=4389)
[   263s] step= 16000  loss=2.8581  fired_acc=0.7347 (n=377)  not_fired_acc=0.7263 (n=4421)
[   295s] step= 18000  loss=2.7979  fired_acc=0.9028 (n=360)  not_fired_acc=0.6631 (n=4438)
[   329s] step= 20000  loss=2.7666  fired_acc=0.9218 (n=371)  not_fired_acc=0.6815 (n=4427)
[   360s] step= 22000  loss=2.7692  fired_acc=0.9431 (n=369)  not

In [15]:
WEIGHT_SWEEP_STEPS = 20000
WEIGHT_SWEEP_VALUES = [2.0, 4.0, 6.0, override_weight]  # override_weight = 11.8, the mass-equalizing value
WEIGHT_SWEEP_SEEDS = (0, 1)

weight_sweep_logs = {}
for w in WEIGHT_SWEEP_VALUES:
    print(f"=== override_weight={w} ===")
    weight_sweep_logs[w] = {}
    for seed in WEIGHT_SWEEP_SEEDS:
        model, log = train_isolated_weighted_split(A_rw, n_embd_tok=96, steps=WEIGHT_SWEEP_STEPS,
                                                     override_weight=w, seed=seed,
                                                     device=DEVICE, eval_every=4000, n_eval=4000)
        weight_sweep_logs[w][seed] = log
        final = log[-1]
        print(f"  seed={seed}  final: fired_acc={final['fired']:.4f}  not_fired_acc={final['not_fired']:.4f}")

import pickle
with open('exp3_weight_sweep.pkl', 'wb') as f:
    pickle.dump(weight_sweep_logs, f)
print("\nsaved exp3_weight_sweep.pkl")

print("\n=== summary: weight -> (fired_acc, not_fired_acc), averaged over seeds ===")
for w in WEIGHT_SWEEP_VALUES:
    fired_vals = [weight_sweep_logs[w][s][-1]['fired'] for s in WEIGHT_SWEEP_SEEDS]
    nf_vals = [weight_sweep_logs[w][s][-1]['not_fired'] for s in WEIGHT_SWEEP_SEEDS]
    print(f"  weight={w:6.1f}   fired_acc={sum(fired_vals)/len(fired_vals):.4f}   "
          f"not_fired_acc={sum(nf_vals)/len(nf_vals):.4f}")

=== override_weight=2.0 ===
[    80s] step=  4000  loss=2.7839  fired_acc=0.0434 (n=392)  not_fired_acc=0.9739 (n=4406)
[   163s] step=  8000  loss=2.7577  fired_acc=0.1094 (n=393)  not_fired_acc=0.9743 (n=4405)
[   245s] step= 12000  loss=2.7367  fired_acc=0.2431 (n=399)  not_fired_acc=0.9645 (n=4399)
[   325s] step= 16000  loss=2.7234  fired_acc=0.3483 (n=379)  not_fired_acc=0.9527 (n=4419)
[   422s] step= 20000  loss=2.7171  fired_acc=0.4401 (n=384)  not_fired_acc=0.9465 (n=4414)
  seed=0  final: fired_acc=0.4401  not_fired_acc=0.9465
[   122s] step=  4000  loss=2.7905  fired_acc=0.3359 (n=384)  not_fired_acc=0.9137 (n=4414)
[   247s] step=  8000  loss=2.7490  fired_acc=0.2418 (n=368)  not_fired_acc=0.9679 (n=4430)
[   371s] step= 12000  loss=2.7333  fired_acc=0.3209 (n=374)  not_fired_acc=0.9521 (n=4424)
[   494s] step= 16000  loss=2.7210  fired_acc=0.4134 (n=387)  not_fired_acc=0.9637 (n=4411)
[   618s] step= 20000  loss=2.7151  fired_acc=0.4343 (n=396)  not_fired_acc=0.9657 (n=44

KeyboardInterrupt: 